# 07 Лаборатория — Продвинутые конструкции: пропорции, бэкспреды, BWB, джейд-лизард

Эта лаборатория строит каждую асимметричную конструкцию на цепочке DEMO (спот 100, 45 DTE) и
показывает, куда переехал риск. Вы:

1. Соберёте колл-ratio 1x2 (пропорциональный спред) и увидите его **безграничный** хвост через
   `analyzer.max_loss`.
2. Соберёте зеркальный колл-бэкспред (backspread) и найдёте его **ограниченную** долину.
3. Сравните бабочку со «сломанным крылом» (broken-wing butterfly) с обычной бабочкой через
   `viz.plot_compare`.
4. Проверите у джейд-лизарда (jade lizard) правило риска вверх: **кредит против ширины
   колл-спреда**.

Работает офлайн, сверху вниз.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, viz, data

SPOT, EXP = 100.0, 45/365
chain = data.load_sample_chain("DEMO")

## 1. Колл-ratio (1x2): голый хвост

Длинный один 100-й колл (call, 3.91), короткие два 105-х колла (по 1.85). Почти в ноль по премии —
но второй короткий колл голый вверх.

In [ ]:
ratio = strategies.call_ratio_spread(
    long_call=(100.0, 3.91), short_call=(105.0, 1.85),
    expiry=EXP, ratio=(1, 2),
)
print(ratio.describe())
print("чистая премия $:", ratio.net_premium())
print("max_profit $:", analyzer.max_profit(ratio))
print("max_loss $:", analyzer.max_loss(ratio))

`max_loss` равен `-inf`: лишний короткий колл делает убытки вверх безграничными. Максимальная
прибыль сидит на коротком страйке (105). Нарисуем выплату на экспирации, чтобы увидеть широкую зону
без убытка и хвост.

In [ ]:
spots = np.linspace(85, 125, 161)
ax = viz.plot_payoff(ratio, spots)
ax.set_title("Колл-ratio 1x2: палатка прибыли, затем безграничный хвост вверх")

## 2. Колл-бэкспред: зеркальное отражение

Короткий один 100-й колл (3.91), длинные два 105-х колла (1.85). Теперь вы **в сумме длинны** по
опционам: ограниченная долина убытка, а затем выпуклая прибыль на большом движении вверх.

In [ ]:
back = strategies.call_backspread(
    short_call=(100.0, 3.91), long_call=(105.0, 1.85),
    expiry=EXP, ratio=(1, 2),
)
print("чистая премия $ (- = кредит):", back.net_premium())
print("max_loss $ (ограничен):", analyzer.max_loss(back))

In [ ]:
pnl = payoff.pnl_curve(back, spots)
valley = spots[np.argmin(pnl)]
print("дно долины около спота:", round(valley, 1), "убыток $:", round(pnl.min(), 2))
ax = viz.plot_payoff(back, spots); ax.set_title("Колл-бэкспред: ограниченная долина, выпуклый рост")

Дно долины приходится на длинный страйк (105); выше него два длинных перебивают один
короткий, и прибыль идёт выпукло. Убыток **ограничен**, потому что голой ноги больше нет.

## 3. Бабочка со «сломанным крылом» против обычной бабочки

Обычная пут-бабочка +1/-2/+1 с равными крыльями по 2.5 (95/97.5/100). BWB с более широким нижним
крылом (92.5/97.5/100), чтобы удешевить дальнюю сторону и перекосить риск.

In [ ]:
reg_fly = strategies.long_put_butterfly(
    low=(95.0, 1.58), mid=(97.5, 2.37), high=(100.0, 3.42), expiry=EXP)
bwb = strategies.broken_wing_butterfly(
    "put", low=(92.5, 1.01), mid=(97.5, 2.37), high=(100.0, 3.42), expiry=EXP)
print("чистая премия обычной бабочки $:", reg_fly.net_premium())
print("чистая премия BWB $:", bwb.net_premium())

In [ ]:
ax = viz.plot_compare([reg_fly, bwb])
ax.set_title("Обычная бабочка (симметричная) против сломанного крыла (перекос, безрисковый верх)")

BWB выполаживается вверху (безрисковая сторона — все путы (put) истекают пустыми, кредит
остаётся у вас) и несёт свой ограниченный убыток на пролёте более широкого нижнего крыла.
Подтвердим анализатором.

In [ ]:
for name, p in [("обычная", reg_fly), ("BWB", bwb)]:
    print(name, "max_profit", round(analyzer.max_profit(p), 1),
          "max_loss", round(analyzer.max_loss(p), 1))

## 4. Джейд-лизард: проверка риска вверх

Короткий 95-й пут (1.58), короткий 105-й колл (1.85), длинный 110-й колл (0.73). Правило: **риска
вверх нет только при суммарном кредите >= ширины колл-спреда**. Проверим это явно.

In [ ]:
jade = strategies.jade_lizard(
    short_put=(95.0, 1.58), short_call=(105.0, 1.85), long_call=(110.0, 0.73), expiry=EXP)
credit = -jade.net_premium() / 100          # на акцию, кредит положителен
width = 110.0 - 105.0
print("кредит на акцию:", round(credit, 2), "ширина колл-спреда:", width)
print("риск вверх устранён?", credit >= width)

Здесь кредит (2.70) < ширины (5.0), значит, у этого лизарда **риск вверх остаётся**.
`max_profit` анализатора на верхнем плато равен кредиту; широкое движение вверх приносит убыток
(ширина − кредит). Изучим выплату и сводку.

In [ ]:
print(analyzer.summarize(jade, SPOT, vol=0.26))
ax = viz.plot_payoff(jade, np.linspace(80, 120, 161))
ax.set_title("Джейд-лизард (кредит < ширины -> риск вверх остаётся)")

## Эксперименты

1. В пропорциональном спреде перенесите короткий страйк на 110 (по 0.73). Хвост становится дешевле
   или опаснее и что происходит с шириной зоны без убытка?
2. Переоцените бэкспред с более низкой плоской `vol` в `analyzer.summarize`. Бэкспреды длинны по
   веге — как более дешёвая волатильность меняет их привлекательность на входе?
3. Сломайте у BWB *верхнее* крыло вместо нижнего (например, low=95, mid=97.5, high=102.5 на коллах).
   Какая сторона теперь безрисковая?
4. Почините риск вверх у джейд-лизарда: сузьте колл-спред до ширины 2.5 (продать 105 / купить 107.5)
   и перепроверьте `credit >= width`. Какой кредит для этого нужен?
5. Наложите пропорциональный спред и бэкспред через `viz.plot_compare` — это зеркальные отражения,
   собранные из одних и тех же трёх страйков.